## Notebook 11 – Date & Time Operations

### Setup
For this notebook I only need pandas. The command below installs it. If it is already installed, pip just says "Requirement already satisfied", so it is safe to run again. I used %pip so that the library is installed in the same environment as this notebook.

The time zone names used later (like Asia/Kolkata) come from a time zone database called tzdata, which is installed together with pandas.

In [1]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\rafiy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from datetime import datetime

Dates and times are used in almost every real dataset: order dates, login times, sensor readings. But when a file is read, dates usually come as plain text, and text cannot be sorted by time, subtracted or grouped by month. In this notebook I learn how pandas handles dates and times.

Note: Depending on the pandas version, the date data type is shown as datetime64[ns] (older versions) or datetime64[us] (pandas 3), and similarly for time differences (timedelta64). It is the same thing, only the precision shown in the name is different.

## Topic 1 – Datetime

### Step 1 – Understand the Concept
A datetime is a value that has a date and, optionally, a time (for example 2024-03-15 14:30:45).

- Python has the datetime class (from the datetime module) to store one such value. It has parts like year, month, day, hour, minute and second.
- Pandas has a special column type for many dates, called datetime64. A column of this type can be sorted by time, filtered by date, and used for date calculations.
- pd.to_datetime() converts text into datetime64.
- pd.date_range() creates a series of dates automatically. The main options are start, end, periods (how many dates) and freq (the step): "D" for days, "B" for business days (Monday to Friday) and "MS" for the first day of every month.
- When the dates are the index of a Series or DataFrame, they can be sliced with date strings like series["2024-01-02":"2024-01-04"].

### Step 2 – Demonstrate the Concept

#### Business Example
A shop keeps the order dates as text like "2024-03-05". As text, the manager cannot find the orders after 1 February or the days between the first and the last order. After the column is converted to datetime64, sorting, filtering and subtracting the dates all work properly.

### Step 3 – Implement the Concept

In [3]:
# One datetime value in Python
py_dt = datetime(2024, 3, 15, 14, 30, 45)
print(py_dt)
print(type(py_dt))
print("Year:", py_dt.year, "| Month:", py_dt.month, "| Day:", py_dt.day, "| Hour:", py_dt.hour)
print("Formatted:", py_dt.strftime("%d-%m-%Y"))

2024-03-15 14:30:45
<class 'datetime.datetime'>
Year: 2024 | Month: 3 | Day: 15 | Hour: 14
Formatted: 15-03-2024


In [4]:
# Dates stored as text
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "order_date": ["2024-03-05", "2024-01-20", "2024-02-14", "2024-03-01", "2024-01-05"],
    "amount": [250, 120, 90, 300, 60]
})
print(orders.dtypes)

# Converting to datetime64
orders["order_date"] = pd.to_datetime(orders["order_date"])
print()
print(orders.dtypes)

order_id      int64
order_date      str
amount        int64
dtype: object

order_id               int64
order_date    datetime64[us]
amount                 int64
dtype: object


In [5]:
# Now the dates can be sorted, filtered and subtracted
print(orders.sort_values("order_date"))

print()
print("Orders on or after 1 February:")
print(orders[orders["order_date"] >= "2024-02-01"])

print()
print("First order :", orders["order_date"].min())
print("Last order  :", orders["order_date"].max())
print("Time between:", orders["order_date"].max() - orders["order_date"].min())

   order_id order_date  amount
4         5 2024-01-05      60
1         2 2024-01-20     120
2         3 2024-02-14      90
3         4 2024-03-01     300
0         1 2024-03-05     250

Orders on or after 1 February:
   order_id order_date  amount
0         1 2024-03-05     250
2         3 2024-02-14      90
3         4 2024-03-01     300

First order : 2024-01-05 00:00:00
Last order  : 2024-03-05 00:00:00
Time between: 60 days 00:00:00


In [6]:
# Creating dates with date_range
print(pd.date_range("2024-01-01", periods=5, freq="D").strftime("%Y-%m-%d").tolist())
print(pd.date_range("2024-01-01", periods=4, freq="MS").strftime("%Y-%m-%d").tolist())
print(pd.date_range("2024-03-01", periods=5, freq="B").strftime("%Y-%m-%d").tolist())
print(len(pd.date_range("2024-01-01", "2024-01-10")))

# Dates as an index: slicing with date strings
daily = pd.Series([10, 12, 9, 15, 11], index=pd.date_range("2024-01-01", periods=5, freq="D"))
print()
print(daily["2024-01-02":"2024-01-04"])

['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']
['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01']
['2024-03-01', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07']
10

2024-01-02    12
2024-01-03     9
2024-01-04    15
Freq: D, dtype: int64


#### What does this program do?
The first cell creates one Python datetime for 15 March 2024, 14:30:45 and prints its parts (year 2024, month 3, day 15, hour 14), and the date in the day-month-year format, 15-03-2024.

In the second cell, the order dates are first stored as text, so the data type of the column is shown as text (str or object). After pd.to_datetime() the type changes to datetime64.

In the third cell, the table is sorted by the date, so the order is 5 (5 January), 2 (20 January), 3 (14 February), 4 (1 March) and 1 (5 March). The filter keeps the orders from 1 February, which are the orders 1, 3 and 4. The first order is on 2024-01-05, the last order is on 2024-03-05, and the difference between them is 60 days (31 days up to 5 February and 29 more days up to 5 March, because 2024 is a leap year).

In the fourth cell, date_range() creates 5 daily dates from 1 to 5 January, 4 monthly dates (1 January, 1 February, 1 March and 1 April), and 5 business days starting from Friday 1 March (1, 4, 5, 6 and 7 March, because the weekend is skipped). A range from 1 to 10 January has 10 dates. When a Series has dates as its index, the slice from "2024-01-02" to "2024-01-04" gives 3 values (12, 9 and 15), and the end date is included.

#### AI/ML Example
Time series data, like daily sales, stock prices or sensor readings, needs a proper datetime column or index. date_range() is used to create a complete calendar of dates, for example to find the days that are missing in the data. Slicing by date strings is used to split the data into a training period and a testing period in the order of time, which is important for time series models.

#### Advantages
- Dates can be sorted, filtered, compared and subtracted correctly.
- Date strings can be used directly in comparisons and slices.
- date_range() creates regular dates with one line.

#### Limitations
- Dates read from files are text at first, so they must be converted before use.
- Wrongly formatted dates can be read in the wrong way (day and month swapped) or cause errors.
- datetime64 cannot store dates that are very far in the past or the future (before the year 1677 or after 2262, in the nanosecond version).

#### Key Point
Convert text dates with pd.to_datetime() so that the column becomes datetime64. Then sorting, filtering and date arithmetic work. date_range() creates dates automatically.

## Topic 2 – Timestamp

### Step 1 – Understand the Concept
A Timestamp is the pandas object for one single date and time. It is the pandas version of the Python datetime, and in fact it is a subclass of it, so it can be used wherever a datetime is used. When we take one value out of a datetime64 column, we get a Timestamp.

It can be created in many ways:
- pd.Timestamp("2024-03-15 14:30:45") from text
- pd.Timestamp(2024, 3, 15) from numbers
- pd.Timestamp.now() for the current date and time

Useful attributes and methods:
- year, month, day, hour, minute, second
- dayofweek (Monday is 0 and Sunday is 6), dayofyear, quarter, days_in_month, is_leap_year
- day_name() and month_name() for the names
- normalize() sets the time to midnight (00:00:00)
- strftime(format) writes the date as text in the format we choose

### Step 2 – Demonstrate the Concept

#### Real-World Example
A bank records the exact moment of a transaction, like 15 March 2024 at 2:30 pm. That one exact moment, with the date and the time, is a Timestamp. From it we can read out the weekday, the month, or the hour.

### Step 3 – Implement the Concept

In [7]:
ts = pd.Timestamp("2024-03-15 14:30:45")
print(ts)
print(type(ts))
print("Is it also a Python datetime?", isinstance(ts, datetime))

print()
print("Year:", ts.year, "| Month:", ts.month, "| Day:", ts.day)
print("Hour:", ts.hour, "| Minute:", ts.minute, "| Second:", ts.second)
print("Day of week (Monday = 0):", ts.dayofweek)
print("Day name   :", ts.day_name())
print("Month name :", ts.month_name())
print("Quarter    :", ts.quarter)
print("Day of year:", ts.dayofyear)
print("Days in month:", ts.days_in_month)
print("Leap year? :", ts.is_leap_year)

2024-03-15 14:30:45
<class 'pandas.Timestamp'>
Is it also a Python datetime? True

Year: 2024 | Month: 3 | Day: 15
Hour: 14 | Minute: 30 | Second: 45
Day of week (Monday = 0): 4
Day name   : Friday
Month name : March
Quarter    : 1
Day of year: 75
Days in month: 31
Leap year? : True


In [8]:
# Other ways to create a Timestamp
print(pd.Timestamp(2024, 3, 15))
print(pd.Timestamp(year=2024, month=3, day=15, hour=9, minute=30))
print("Right now:", pd.Timestamp.now())

# Changing and formatting
print()
print("normalize :", ts.normalize())
print("replace   :", ts.replace(hour=9, minute=0, second=0))
print(ts.strftime("%d-%m-%Y"))
print(ts.strftime("%B %d, %Y"))
print(ts.strftime("%A, %I:%M %p"))

# Comparing
print()
print(ts > pd.Timestamp("2024-01-01"))
print(ts == pd.Timestamp("2024-03-15 14:30:45"))

2024-03-15 00:00:00
2024-03-15 09:30:00
Right now: 2026-10-16 15:54:55.658334

normalize : 2024-03-15 00:00:00
replace   : 2024-03-15 09:00:00
15-03-2024
March 15, 2024
Friday, 02:30 PM

True
True


#### What does this program do?
In the first cell, a Timestamp is created from the text "2024-03-15 14:30:45". The type is pandas Timestamp, and isinstance() gives True, which shows that it is also a normal Python datetime. The parts are year 2024, month 3, day 15, hour 14, minute 30 and second 45. The day of the week is 4 (Monday is 0, so Friday is 4), the day name is Friday and the month name is March. The quarter is 1, the day of the year is 75 (31 days of January, 29 of February and 15 of March), the month has 31 days and 2024 is a leap year, so it prints True.

In the second cell, a Timestamp is created in two other ways, and Timestamp.now() gives the current date and time of the computer, so this value is different every time the cell is run. normalize() sets the time to midnight (2024-03-15 00:00:00) and replace() changes some parts (the time becomes 09:00:00). strftime() writes the date in different formats: 15-03-2024, March 15, 2024 and Friday, 02:30 PM. At the end the comparisons give True and True.

#### AI/ML Example
Event logs, like the clicks of users on a website or the readings of a machine sensor, have a timestamp for every event. From a single Timestamp we can read out features such as the hour of the day, the day of the week and the month, which help to find patterns, for example that more clicks happen in the evening or on weekends.

#### Advantages
- Gives easy access to every part of a date and time.
- Works like a normal Python datetime, so it can be used with other libraries.
- strftime() gives full control over how the date is written.

#### Limitations
- Timestamp.now() changes every time, so results that depend on it cannot be repeated.
- A Timestamp is only one value. For a whole column, the dt accessor has to be used (explained in Topic 6).
- Without a time zone, it does not say in which part of the world the moment happened.

#### Key Point
A Timestamp is one exact date and time in pandas. It has attributes like year, month, dayofweek and methods like day_name() and strftime().

## Topic 3 – Date Parsing

### Step 1 – Understand the Concept
Date parsing means converting text (or numbers) into real dates. The function is pd.to_datetime(), and the important options are:

- format : tells exactly how the text is written. This is the safest way and also the fastest.
- dayfirst=True : read the first number as the day (for 03/04/2024 it gives 3 April and not 4 March).
- errors : "raise" (default) gives an error for a bad date, "coerce" turns bad dates into NaT (Not a Time, the missing value of dates).
- unit : used for numbers, like unit="s" for seconds since 1 January 1970 (Unix time).
- A table with the columns year, month and day can be converted directly with pd.to_datetime(df[["year", "month", "day"]]).
- read_csv() can also parse dates while reading with parse_dates=["column"].

Common format codes:

| Code | Meaning | Example |
|---|---|---|
| %Y | year with 4 digits | 2024 |
| %y | year with 2 digits | 24 |
| %m | month as a number | 03 |
| %B | full month name | March |
| %b | short month name | Mar |
| %d | day of the month | 15 |
| %H | hour (24 hour clock) | 14 |
| %I | hour (12 hour clock) | 02 |
| %M | minute | 30 |
| %S | second | 45 |
| %p | AM or PM | PM |
| %A | full weekday name | Friday |

### Step 2 – Demonstrate the Concept

#### Business Example
The staff of a company type the dates in the day/month/year style, like 03/04/2024 for 3 April. If pandas reads this as month/day/year, it becomes 4 March, which is wrong and does not give any error. The right format (or dayfirst=True) must be given to read the dates correctly.

### Step 3 – Implement the Concept

In [11]:
# Dates in the ISO style (year-month-day)
print(pd.to_datetime(pd.Series(["2024-03-15", "2024-04-01", "2024-12-25"])))

# Dates in the day/month/year style
text_dates = pd.Series(["15/03/2024", "01/04/2024", "25/12/2024"])
print()
print(pd.to_datetime(text_dates, format="%d/%m/%Y"))

# Dates with month names
print()
print(pd.to_datetime(pd.Series(["15 Mar 2024", "20 Apr 2024"]), format="%d %b %Y"))

0   2024-03-15
1   2024-04-01
2   2024-12-25
dtype: datetime64[us]

0   2024-03-15
1   2024-04-01
2   2024-12-25
dtype: datetime64[us]

0   2024-03-15
1   2024-04-20
dtype: datetime64[us]


In [12]:
# The same text can mean two different dates
print("Default      :", pd.to_datetime("03/04/2024"))
print("dayfirst=True:", pd.to_datetime("03/04/2024", dayfirst=True))
print("With format  :", pd.to_datetime("03/04/2024", format="%d/%m/%Y"))

Default      : 2024-03-04 00:00:00
dayfirst=True: 2024-04-03 00:00:00
With format  : 2024-04-03 00:00:00


In [13]:
# Bad dates
messy = pd.Series(["2024-03-15", "not a date", "2024-13-45"])

try:
    pd.to_datetime(messy)
except ValueError:
    print("ValueError: some values cannot be read as dates")

parsed = pd.to_datetime(messy, errors="coerce")
print()
print(parsed)
print("Number of NaT:", parsed.isna().sum())

ValueError: some values cannot be read as dates

0   2024-03-15
1          NaT
2          NaT
dtype: datetime64[us]
Number of NaT: 2


In [14]:
# Numbers: Unix time (seconds since 1 January 1970)
print(pd.to_datetime(1710513045, unit="s"))

# Separate year, month and day columns
parts = pd.DataFrame({"year": [2024, 2024, 2025], "month": [3, 4, 1], "day": [15, 1, 31]})
print()
print(pd.to_datetime(parts))

# Parsing while reading a file: read_csv(..., parse_dates=["order_date"])

2024-03-15 14:30:45

0   2024-03-15
1   2024-04-01
2   2025-01-31
dtype: datetime64[us]


#### What does this program do?
In the first cell, the ISO style dates are read without any extra option, and give the dates 2024-03-15, 2024-04-01 and 2024-12-25 with the type datetime64. The day/month/year dates need format="%d/%m/%Y", and give the same three dates (15 March, 1 April and 25 December). The dates with month names are read with the format "%d %b %Y", and give 2024-03-15 and 2024-04-20.

The second cell shows the danger of the text 03/04/2024. By default pandas reads it as month first, so it gives 2024-03-04 (4 March). With dayfirst=True, or with the format "%d/%m/%Y", it gives 2024-04-03 (3 April).

In the third cell, the Series has a good date and two bad values (a sentence and a date with the month 13). The default gives a ValueError, so the message is printed. With errors="coerce" the bad values become NaT, so the result is 2024-03-15, NaT and NaT, and the number of NaT is 2.

In the fourth cell, the number 1710513045 with unit="s" is read as seconds since 1 January 1970, and gives 2024-03-15 14:30:45. The table with the columns year, month and day is converted directly into the dates 2024-03-15, 2024-04-01 and 2025-01-31. The comment at the end shows how to parse dates while reading a CSV file.

#### AI/ML Example
Dates in files are read as text, so one of the first steps of data preparation is to parse them with pd.to_datetime(). With errors="coerce" the wrong dates become NaT, and they can be counted and handled like other missing values. Unix times (numbers) are very common in logs and data from apps, and they are converted with the unit option before features are made.

#### Advantages
- One function reads many different date styles.
- The format option makes parsing exact and fast.
- errors="coerce" prevents one bad value from stopping the whole conversion.

#### Limitations
- A date like 03/04/2024 is ambiguous, and pandas can read it wrongly without an error if the format is not given.
- Without the format option, parsing of big columns can be slow, and mixed styles in one column can cause errors.
- With errors="coerce", bad dates become NaT silently, so the count should be checked.

#### Key Point
Use pd.to_datetime(column, format=...) to parse dates. Give the format for day/month dates, and use errors="coerce" to turn bad values into NaT.

## Topic 4 – Timedelta

### Step 1 – Understand the Concept
A Timedelta is a length of time, like "3 days" or "8 hours 30 minutes". It is what we get when two dates are subtracted, and it can also be added to or subtracted from dates.

- Date minus date gives a Timedelta.
- Date plus a Timedelta gives a new date.
- pd.Timedelta("2 days 3 hours") or pd.Timedelta(days=5, hours=12) creates one.
- For a whole column, the dt accessor gives the parts: .dt.days (whole days) and .dt.total_seconds() (the full length in seconds).
- Timedeltas can be compared, averaged, summed and so on.
- pd.DateOffset(months=1) moves by calendar units (months or years), which are not always the same number of days. A Timedelta is always an exact length of time.

### Step 2 – Demonstrate the Concept

#### Business Example
An online shop wants to know how many days each order took to be delivered. The delivery time is the delivery date minus the order date, and the result is a Timedelta for every order. The orders that took more than 3 days can be found by comparing with a Timedelta.

### Step 3 – Implement the Concept

In [15]:
deliveries = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "order_date": pd.to_datetime(["2024-03-01", "2024-03-02", "2024-03-05", "2024-03-10", "2024-03-12"]),
    "delivery_date": pd.to_datetime(["2024-03-04", "2024-03-07", "2024-03-06", "2024-03-17", "2024-03-15"])
})

# Date minus date gives a Timedelta
deliveries["delivery_time"] = deliveries["delivery_date"] - deliveries["order_date"]
deliveries["days"] = deliveries["delivery_time"].dt.days
print(deliveries)

print()
print("Average delivery time:", deliveries["delivery_time"].mean())
print("Longest delivery time:", deliveries["delivery_time"].max())

# Late orders: more than 3 days
print()
print(deliveries[deliveries["delivery_time"] > pd.Timedelta(days=3)])

   order_id order_date delivery_date delivery_time  days
0         1 2024-03-01    2024-03-04        3 days     3
1         2 2024-03-02    2024-03-07        5 days     5
2         3 2024-03-05    2024-03-06        1 days     1
3         4 2024-03-10    2024-03-17        7 days     7
4         5 2024-03-12    2024-03-15        3 days     3

Average delivery time: 3 days 19:12:00
Longest delivery time: 7 days 00:00:00

   order_id order_date delivery_date delivery_time  days
1         2 2024-03-02    2024-03-07        5 days     5
3         4 2024-03-10    2024-03-17        7 days     7


In [16]:
# Creating Timedeltas
t1 = pd.Timedelta("2 days 3 hours")
t2 = pd.Timedelta(days=5, hours=12)
print(t1, "->", t1.total_seconds(), "seconds")
print(t2, "->", t2.total_seconds(), "seconds")

# The difference between two times of the same day
work = pd.Timestamp("2024-03-15 18:00") - pd.Timestamp("2024-03-15 09:30")
print()
print("Working time:", work)
print("In seconds  :", work.total_seconds())
print("In hours    :", work.total_seconds() / 3600)

# Adding a Timedelta to a date
print()
print(pd.Timestamp("2024-03-15") + pd.Timedelta(days=30))
print((deliveries["order_date"] + pd.Timedelta(days=2)).dt.strftime("%Y-%m-%d").tolist())

2 days 03:00:00 -> 183600.0 seconds
5 days 12:00:00 -> 475200.0 seconds

Working time: 0 days 08:30:00
In seconds  : 30600.0
In hours    : 8.5

2024-04-14 00:00:00
['2024-03-03', '2024-03-04', '2024-03-07', '2024-03-12', '2024-03-14']


In [17]:
# Timedelta and DateOffset are different
start = pd.Timestamp("2024-01-31")
print("Plus 30 days   :", start + pd.Timedelta(days=30))
print("Plus 1 month   :", start + pd.DateOffset(months=1))

# ML-style: days since the order (recency)
reference_date = pd.Timestamp("2024-04-01")
deliveries["days_since_order"] = (reference_date - deliveries["order_date"]).dt.days
print()
print(deliveries[["order_id", "order_date", "days_since_order"]])

Plus 30 days   : 2024-03-01 00:00:00
Plus 1 month   : 2024-02-29 00:00:00

   order_id order_date  days_since_order
0         1 2024-03-01                31
1         2 2024-03-02                30
2         3 2024-03-05                27
3         4 2024-03-10                22
4         5 2024-03-12                20


#### What does this program do?
In the first cell, the delivery time is calculated by subtracting the order date from the delivery date, which gives a Timedelta column, and .dt.days gives the number of days. The delivery times are 3, 5, 1, 7 and 3 days. The average delivery time is 3 days 19:12:00, which is 3.8 days, and the longest is 7 days. Comparing with pd.Timedelta(days=3) finds the orders that took more than 3 days, which are order 2 (5 days) and order 4 (7 days).

In the second cell, the Timedelta "2 days 3 hours" is 183600 seconds, and 5 days 12 hours is 475200 seconds. The difference between 18:00 and 09:30 is 8 hours 30 minutes, which is 30600 seconds or 8.5 hours. Adding 30 days to 15 March 2024 gives 2024-04-14, and adding 2 days to the order dates gives the expected delivery dates 2024-03-03, 2024-03-04, 2024-03-07, 2024-03-12 and 2024-03-14.

In the third cell, 30 days after 31 January 2024 is 2024-03-01, but one month after 31 January is 2024-02-29 with DateOffset, because it moves by calendar months and stops at the last day of February (2024 is a leap year). This shows that a month is not the same as 30 days. At the end, the days from each order to 1 April 2024 are calculated, and the values are 31, 30, 27, 22 and 20.

#### AI/ML Example
Time differences are very useful features. Examples are recency (days since the last purchase of a customer), the age of an account, the time between two events, and the delivery time in the example above. Models cannot use dates directly, but they can use these numbers.

#### Advantages
- Date arithmetic works directly, with no manual calculation.
- Gives the difference in days, hours or seconds.
- Creates useful numeric features from dates.

#### Limitations
- .dt.days gives only whole days and cuts off the rest, so total_seconds() is needed for exact values.
- A Timedelta cannot express months or years exactly, because these have different lengths (DateOffset is needed for that).
- Subtracting dates with and without a time zone gives an error.

#### Key Point
Date minus date gives a Timedelta, and date plus Timedelta gives a date. Use .dt.days or total_seconds() to get numbers, and DateOffset for calendar months and years.

## Topic 5 – Time Zones

### Step 1 – Understand the Concept
A time zone says to which part of the world a time belongs.

- A naive datetime has no time zone, so it is not known which place the time belongs to.
- A time zone aware datetime has a time zone, like 2024-03-15 10:00:00+05:30.
- tz_localize("zone") gives a time zone to a naive time. It does not change the clock time, it only says where the time belongs. tz_localize(None) removes the time zone.
- tz_convert("zone") changes an aware time to another zone. The moment is the same, and only the clock time is different.
- For a column, the dt accessor is used: .dt.tz_localize() and .dt.tz_convert().
- Zone names look like "Asia/Kolkata", "America/New_York" or "UTC". UTC is the standard world time, and it is the best zone for storing times.
- Two aware times from different zones can be compared and subtracted correctly, but a naive time and an aware time cannot be subtracted.

### Step 2 – Demonstrate the Concept

#### Business Example
A company in India fixes a meeting for 10:00 am on 15 March 2024 Indian time. The colleagues in New York and Tokyo need to know their own local time for the same meeting. The time is first given the Indian zone (localize) and then converted to the other zones.

### Step 3 – Implement the Concept

In [18]:
meeting = pd.Timestamp("2024-03-15 10:00")
print("Without a time zone:", meeting, "| tz =", meeting.tz)

# Giving the Indian time zone
ist = meeting.tz_localize("Asia/Kolkata")
print("India  :", ist)

# Converting to other zones (the same moment)
print("UTC    :", ist.tz_convert("UTC"))
print("New York:", ist.tz_convert("America/New_York"))
print("Tokyo  :", ist.tz_convert("Asia/Tokyo"))

# Same moment in two zones
ny = ist.tz_convert("America/New_York")
print()
print("Same moment?", ist == ny)
print("Difference  :", ist - ny)

Without a time zone: 2024-03-15 10:00:00 | tz = None
India  : 2024-03-15 10:00:00+05:30
UTC    : 2024-03-15 04:30:00+00:00
New York: 2024-03-15 00:30:00-04:00
Tokyo  : 2024-03-15 13:30:00+09:00

Same moment? True
Difference  : 0 days 00:00:00


In [19]:
# The meeting time in many zones
utc_time = ist.tz_convert("UTC")
for zone in ["UTC", "Asia/Kolkata", "America/New_York", "Asia/Tokyo", "Europe/London"]:
    print(zone.ljust(18), utc_time.tz_convert(zone).strftime("%Y-%m-%d %H:%M"))

# Removing the time zone again
print()
print(utc_time.tz_localize(None))

UTC                2024-03-15 04:30
Asia/Kolkata       2024-03-15 10:00
America/New_York   2024-03-15 00:30
Asia/Tokyo         2024-03-15 13:30
Europe/London      2024-03-15 04:30

2024-03-15 04:30:00


In [20]:
# A whole column
meetings = pd.to_datetime(pd.Series(["2024-03-15 10:00", "2024-03-15 15:30"]))
meetings_ist = meetings.dt.tz_localize("Asia/Kolkata")
meetings_utc = meetings_ist.dt.tz_convert("UTC")
print(meetings_ist)
print()
print(meetings_utc)

# A naive time and an aware time cannot be mixed
naive = pd.Timestamp("2024-03-15 10:00")
aware = naive.tz_localize("UTC")
try:
    aware - naive
except TypeError as error:
    print()
    print("TypeError:", error)

0   2024-03-15 10:00:00+05:30
1   2024-03-15 15:30:00+05:30
dtype: datetime64[us, Asia/Kolkata]

0   2024-03-15 04:30:00+00:00
1   2024-03-15 10:00:00+00:00
dtype: datetime64[us, UTC]

TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects.


#### What does this program do?
In the first cell, the meeting time has no time zone at first (tz is None). tz_localize("Asia/Kolkata") says that the time is the Indian time, and gives 2024-03-15 10:00:00+05:30, so the clock time is not changed. After that, tz_convert() gives the same moment in other zones: 04:30 in UTC (+00:00), 00:30 on the same day in New York (-04:00, because summer time has already started in the USA on 10 March) and 13:30 in Tokyo (+09:00). Comparing the Indian time with the New York time gives True, and the difference is 0 days 00:00:00, because it is the same moment.

In the second cell, the loop prints the time in 5 zones: 04:30 in UTC, 10:00 in Kolkata, 00:30 in New York, 13:30 in Tokyo and 04:30 in London (London still has the winter time on 15 March). tz_localize(None) removes the zone and leaves the UTC clock time, 2024-03-15 04:30:00.

In the third cell, two meeting times are given the Indian zone as a column, and then converted to UTC: 10:00 becomes 04:30 and 15:30 becomes 10:00. At the end, subtracting a naive time from an aware time gives a TypeError about mixing tz-naive and tz-aware values.

#### AI/ML Example
Data collected from servers or users in different countries has times in different zones. Before calculating features like the hour of the day or the time between events, all the times must be converted to one zone, usually UTC. Otherwise, the same event can look like it happened at different hours, or the time differences can be wrong.

#### Advantages
- Times from different parts of the world can be compared correctly.
- Daylight saving time changes are handled automatically by the time zone database.
- Storing everything in UTC avoids confusion.

#### Limitations
- Naive and aware times cannot be mixed, which gives an error.
- Using the wrong zone name in tz_localize() gives a wrong time silently.
- Local times that do not exist or happen twice when the clock changes (daylight saving) need special handling.

#### Key Point
tz_localize() gives a time zone to a naive time, and tz_convert() changes an aware time to another zone. Store times in UTC.

## Topic 6 – Extracting Date Features

### Step 1 – Understand the Concept
For a whole datetime column, the parts of the dates are taken out with the dt accessor, like df["date"].dt.month. Each of them gives a new column.

| Feature | What it gives |
|---|---|
| dt.year, dt.month, dt.day | year, month and day as numbers |
| dt.hour, dt.minute, dt.second | the time parts |
| dt.dayofweek | day of the week (Monday is 0 and Sunday is 6) |
| dt.day_name(), dt.month_name() | names of the weekday and the month |
| dt.quarter | quarter of the year (1 to 4) |
| dt.dayofyear | day number in the year (1 to 366) |
| dt.isocalendar().week | week number of the year |
| dt.is_month_start, dt.is_month_end | True or False |
| dt.strftime(format) | the date as text in the chosen format |

A weekend flag can be made with dayofweek >= 5. These new columns can be used for grouping, filtering and as features for a model.

### Step 2 – Demonstrate the Concept

#### Business Example
A shop has a log of its sales with the date and time of each sale. The owner wants to know in which months, on which days of the week and at which hours the shop sells more. For this, the month, the weekday and the hour are taken out of the date column and used for grouping.

### Step 3 – Implement the Concept

In [21]:
log = pd.DataFrame({
    "timestamp": pd.to_datetime([
        "2024-03-01 09:15", "2024-03-02 14:40", "2024-03-04 11:05", "2024-03-08 19:30",
        "2024-03-31 22:10", "2024-04-01 08:00", "2024-12-25 10:45", "2024-02-29 16:20"
    ]),
    "amount": [120, 200, 150, 300, 80, 90, 400, 60]
})

ts_col = log["timestamp"]
features = pd.DataFrame({
    "timestamp": ts_col,
    "year": ts_col.dt.year,
    "month": ts_col.dt.month,
    "day": ts_col.dt.day,
    "hour": ts_col.dt.hour,
    "dayofweek": ts_col.dt.dayofweek,
    "day_name": ts_col.dt.day_name(),
    "quarter": ts_col.dt.quarter,
    "dayofyear": ts_col.dt.dayofyear,
    "week": ts_col.dt.isocalendar().week
})
print(features)

            timestamp  year  month  day  hour  dayofweek   day_name  quarter  \
0 2024-03-01 09:15:00  2024      3    1     9          4     Friday        1   
1 2024-03-02 14:40:00  2024      3    2    14          5   Saturday        1   
2 2024-03-04 11:05:00  2024      3    4    11          0     Monday        1   
3 2024-03-08 19:30:00  2024      3    8    19          4     Friday        1   
4 2024-03-31 22:10:00  2024      3   31    22          6     Sunday        1   
5 2024-04-01 08:00:00  2024      4    1     8          0     Monday        2   
6 2024-12-25 10:45:00  2024     12   25    10          2  Wednesday        4   
7 2024-02-29 16:20:00  2024      2   29    16          3   Thursday        1   

   dayofyear  week  
0         61     9  
1         62     9  
2         64    10  
3         68    10  
4         91    13  
5         92    14  
6        360    52  
7         60     9  


In [22]:
# More features: weekend, start and end of the month
log["is_weekend"] = log["timestamp"].dt.dayofweek >= 5
log["is_month_start"] = log["timestamp"].dt.is_month_start
log["is_month_end"] = log["timestamp"].dt.is_month_end
print(log)

# The date as text in another format
print()
print(log["timestamp"].dt.strftime("%d-%b-%Y").tolist())
print(log["timestamp"].dt.month_name().tolist())

            timestamp  amount  is_weekend  is_month_start  is_month_end
0 2024-03-01 09:15:00     120       False            True         False
1 2024-03-02 14:40:00     200        True           False         False
2 2024-03-04 11:05:00     150       False           False         False
3 2024-03-08 19:30:00     300       False           False         False
4 2024-03-31 22:10:00      80        True           False          True
5 2024-04-01 08:00:00      90       False            True         False
6 2024-12-25 10:45:00     400       False           False         False
7 2024-02-29 16:20:00      60       False           False          True

['01-Mar-2024', '02-Mar-2024', '04-Mar-2024', '08-Mar-2024', '31-Mar-2024', '01-Apr-2024', '25-Dec-2024', '29-Feb-2024']
['March', 'March', 'March', 'March', 'March', 'April', 'December', 'February']


In [23]:
# Using the features: total sales for each month
print(log.groupby(log["timestamp"].dt.month)["amount"].sum())

# Sales in the evening (hour 18 or later)
print()
print(log[log["timestamp"].dt.hour >= 18])

# Sales on the weekend
print()
print("Weekend sales:", log[log["is_weekend"]]["amount"].sum())

timestamp
2      60
3     850
4      90
12    400
Name: amount, dtype: int64

            timestamp  amount  is_weekend  is_month_start  is_month_end
3 2024-03-08 19:30:00     300       False           False         False
4 2024-03-31 22:10:00      80        True           False          True

Weekend sales: 280


#### What does this program do?
In the first cell, the sales log has 8 sales with a date and time. The dt accessor takes out year, month, day, hour, day of the week, day name, quarter, day of the year and week number as new columns. For example, the first sale on 2024-03-01 09:15 is month 3, hour 9, day of the week 4 (Friday), quarter 1, day of the year 61 and week 9. The sale on 2024-03-31 is a Sunday (day of the week 6, week 13), the sale on 2024-04-01 is a Monday in quarter 2 (day of the year 92, week 14), Christmas 2024-12-25 is a Wednesday in quarter 4 (day of the year 360, week 52) and 2024-02-29 is a Thursday (day of the year 60, week 9).

In the second cell, three True and False columns are added. is_weekend is True for the two sales on Saturday 2 March and Sunday 31 March. is_month_start is True for 1 March and 1 April, and is_month_end is True for 31 March and 29 February. strftime("%d-%b-%Y") writes the dates like 01-Mar-2024, and month_name() gives names like March.

In the third cell, the sales are grouped by month. The totals are February 60, March 850 (120 + 200 + 150 + 300 + 80), April 90 and December 400. The sales at 18:00 or later are the two sales at 19:30 on 8 March and 22:10 on 31 March. The weekend sales are 200 + 80 = 280.

#### AI/ML Example
A model cannot use a raw date, so the date is split into useful numeric features: the hour, the day of the week, the month, the quarter and whether it is a weekend. These features help a model to find patterns such as higher sales on Fridays and weekends, or in December. This is called feature engineering with dates.

#### Advantages
- Creates many useful features from one date column.
- Works on the whole column at once, with no loop.
- The features can be used for grouping, filtering and modelling.

#### Limitations
- The dt accessor works only on datetime64 columns, so text dates must be converted first (otherwise there is an AttributeError).
- Numbers like the month or the hour are cyclical (December is next to January, 23:00 is next to 00:00), but a model sees them as far apart unless they are encoded in a special way.
- The week number from isocalendar() can belong to the previous or the next year at the start or the end of the year.

#### Key Point
Use the dt accessor, like df["date"].dt.month or df["date"].dt.dayofweek, to take date features out of a datetime column. Monday is 0 in dayofweek.

## Interview Questions

1. How do you convert a column of text dates to real dates in pandas? What is the difference between datetime, Timestamp and datetime64?

Use pd.to_datetime(df["col"]), and give format="..." for a known style. A datetime is the Python class for one date and time, a Timestamp is the pandas version of it (it is a subclass of datetime), and datetime64 is the data type of a pandas column that holds many such values. A single value taken from a datetime64 column is a Timestamp.

2. What do errors="coerce", format and dayfirst do in pd.to_datetime()?

format tells exactly how the text is written (like "%d/%m/%Y"), which is the safest and the fastest way. dayfirst=True reads the first number as the day, so 03/04/2024 becomes 3 April. errors="coerce" turns the values that cannot be read into NaT instead of raising an error, so the bad values can be counted and handled.

3. How do you find the difference between two dates in days?

Subtract the two date columns, which gives a Timedelta column, and then use .dt.days for whole days, or .dt.total_seconds() for the exact length in seconds (divide by 86400 for days). For example, (df["delivery_date"] - df["order_date"]).dt.days.

4. What is the difference between tz_localize() and tz_convert()? Why is it good to store times in UTC?

tz_localize() gives a time zone to a naive time without changing the clock time, and tz_convert() changes a time that already has a zone to another zone (the same moment, different clock time). Storing times in UTC avoids confusion with daylight saving and different zones, so times from different places can be compared and subtracted correctly.

5. How do you take features out of a date column, and why is it useful for machine learning?

Use the dt accessor on a datetime64 column: dt.year, dt.month, dt.day, dt.hour, dt.dayofweek (Monday is 0), dt.quarter, dt.day_name() and so on. A model cannot use a raw date, but these numeric features (hour, weekday, month, is_weekend) show patterns related to time, like more sales on weekends or in December.